# 05 — Model Training

**AttriSense · Employee Attrition Prediction & Analytics System**

---

## Purpose

Train and tune four supervised classifiers to predict employee attrition. All training logic lives in `src/attrisense/models/`; this notebook orchestrates the run and displays computed results.

**Models:** Logistic Regression · Decision Tree · Random Forest · XGBoost

**Protocol:**
- Stratified 80/20 train/test split (`random_state=42`)
- 5-fold stratified cross-validation for hyperparameter tuning
- ROC-AUC as the CV scoring metric (appropriate for imbalanced data)
- Sklearn Pipelines: preprocessing + classifier fit together
- Best pipeline per model saved with Joblib

**Inputs:** `employee_attrition_featured.parquet`, `models/selected_features.json`

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from attrisense.config import load_config
from attrisense.models import (
    get_model_specs,
    prepare_training_data,
    run_training_pipeline,
    stratified_train_test_split,
)

config = load_config()

x, y, features = prepare_training_data(config)
x_train, x_test, y_train, y_test = stratified_train_test_split(x, y, config)

print(f"Features   : {len(features)}")
print(f"Train set  : {len(x_train):,} ({y_train.mean()*100:.1f}% attrition)")
print(f"Test set   : {len(x_test):,} ({y_test.mean()*100:.1f}% attrition)")
print(f"Random seed: {config.random_state}")

## 1. Model Pipelines and Hyperparameter Grids

Each model uses a sklearn `Pipeline`: **preprocessor** (one-hot encoding for nominal columns; scaling only for Logistic Regression) followed by **classifier**.

Cross-validation tuning searches the grids below. Grids are intentionally small to keep runtime reasonable on ~1,200 training rows.

In [ ]:
specs = get_model_specs(config)

for name, spec in specs.items():
    print(f"\n{name}")
    print(f"  Scale numeric: {spec['scale_numeric']}")
    print(f"  Param grid   : {spec['param_grid']}")

## 2. Train All Models

`GridSearchCV` with stratified 5-fold CV runs on the **training split only**. The test set is held out until after tuning completes.

This cell may take a few minutes — all metrics are computed at runtime.

In [ ]:
report = run_training_pipeline(config)

print(f"Training complete — {len(report.results)} models saved")
print(f"Results JSON: {report.results_path}")
report.comparison

## 3. Test-Set Performance Comparison

Hold-out metrics on the 20% test split. ROC-AUC measures ranking quality; recall on the attrition class reflects how many leavers the model catches.

In [ ]:
comparison = report.comparison.copy()
metrics = ["test_roc_auc", "test_f1", "test_recall", "test_precision"]

fig, ax = plt.subplots(figsize=(9, 4))
x_pos = range(len(comparison))
width = 0.2

for i, metric in enumerate(metrics):
    offset = (i - 1.5) * width
    ax.bar(
        [p + offset for p in x_pos],
        comparison[metric],
        width=width,
        label=metric.replace("test_", "").upper(),
    )

ax.set_xticks(list(x_pos))
ax.set_xticklabels(comparison["model"], rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Hold-Out Test Metrics by Model")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.15))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Best Hyperparameters

Winning configuration from cross-validation for each model.

In [ ]:
best_params = pd.DataFrame(
    [
        {
            "model": r.model_name,
            "cv_roc_auc": round(r.cv_best_score, 4),
            "best_params": json.dumps(r.best_params),
        }
        for r in report.results
    ]
)
best_params

## 5. Persisted Artifacts

Each saved pipeline includes preprocessing and the tuned classifier — ready for inference without re-fitting.

In [ ]:
artifacts = pd.DataFrame(
    [{"model": r.model_name, "path": r.model_path} for r in report.results]
)
artifacts

**Next step:** `06_Model_Evaluation.ipynb` — confusion matrices, ROC curves, SHAP explanations, and final model selection for deployment.